# 03 · Phase 2: train the visual branch
EfficientNet-B0 (ImageNet-pretrained) fine-tuned on the face crops from notebook 02.

- **Runtime:** `Runtime → Change runtime type → T4 GPU`.
- **Needs:** `processed/<dataset>/index.csv` on Drive (notebook 02). Raw data is *not* needed.
- **Disconnected?** Re-run the setup cell and the training cell: training **resumes** from `last.pt`.

In [ ]:
# --- Session setup: run this first, every time you open the notebook ---
REPO_URL = "https://github.com/tanishq1101/robust-multimodal-deepfake.git"  # "" = use code copied to Drive

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

if REPO_URL:
    CODE = "/content/DeepFake"
    if os.path.exists(CODE):
        subprocess.run(["git", "-C", CODE, "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, CODE], check=True)
else:
    # No GitHub repo yet: upload the project folder to Google Drive at MyDrive/deepfake_project/code
    CODE = "/content/drive/MyDrive/deepfake_project/code"
    assert os.path.exists(CODE), f"Upload the project folder to {CODE} or set REPO_URL"

os.chdir(CODE)
!pip install -q -r requirements-colab.txt && pip install -q -e .
sys.path.insert(0, f"{CODE}/src")

from deepfake.config import load_config
cfg = load_config()
print("data_root (Drive):", cfg["data_root"])
print("raw data (VM disk):", cfg["paths"]["raw"])

In [ ]:
DATASET = "dfdc"          # "fakeavceleb" once it is downloaded + preprocessed
RUN = f"visual_{DATASET}"
EPOCHS = 10
BATCH = 8                 # videos per batch (x 8 frames each); lower to 4 if CUDA runs out of memory

## Train (+ automatic test-set evaluation of the best checkpoint)

In [ ]:
!python -m deepfake.training.train_visual --dataset $DATASET --run-name $RUN --epochs $EPOCHS --batch-size $BATCH --workers 2

## Training curves

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display

run_dir = Path(cfg["paths"]["checkpoints"]) / RUN
display(Image(str(run_dir / "curves.png")))   # saved by the training script
pd.read_csv(run_dir / "history.csv")

## Test results

In [ ]:
m = json.loads((run_dir / "test" / "metrics.json").read_text())
print({k: round(v, 4) if isinstance(v, float) else v for k, v in m.items() if k not in ("index", "checkpoint")})
display(Image(str(run_dir / "test" / "roc.png")))

In [ ]:
# Score distribution: a good model pushes real towards 0 and fake towards 1
p = pd.read_csv(run_dir / "test" / "predictions.csv")
plt.figure(figsize=(6, 3))
for lab, name in [(0, "real"), (1, "fake")]:
    plt.hist(p[p.label == lab].score, bins=20, range=(0, 1), alpha=0.6, label=name)
plt.xlabel("P(fake)"); plt.legend(); plt.tight_layout()

## Cross-dataset check (optional)
Evaluate this model on another processed dataset, e.g. Celeb-DF, to measure generalisation.

In [ ]:
OTHER = "celebdf"
import os
if os.path.exists(f"{cfg['paths']['processed']}/{OTHER}/index.csv"):
    !python -m deepfake.evaluation.evaluate --checkpoint {run_dir}/best.pt --dataset $OTHER --split test